# SparseChron — D-NeRF "mutant" 4D Viewer + Eval (NO training)

Same as the main training notebook but **training is replaced by downloading the trained checkpoint from the GitHub release** — no Kaggle datasets to attach, no training to run.

Run cells top to bottom. Cell 3 fetches the retrained model (PSNR 29.17 / SSIM 0.945 / LPIPS 0.083). Cell 3b optionally switches to the August model (347k Gaussians, dramatic motion) — re-run the last viewer cell after it.

Requires: **Internet ON**, **GPU T4 ON** (Session options).


In [ ]:
# 1. Clone repo + install dependencies
#    Tries a PREBUILT gsplat wheel first (~1 min). If none matches this
#    environment, falls back to building from source (10-15 min, silent = normal).
import os, subprocess, sys

REPO = "/kaggle/working/SparseChron"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/rajhodedara/SparseChron.git", REPO], check=True)
else:
    print("Repo already cloned, skipping.")

%cd /kaggle/working/SparseChron

# fast path: match a prebuilt gsplat wheel to this env's torch + CUDA versions
import torch
tv = torch.__version__.split("+")[0]
cu = (torch.version.cuda or "").replace(".", "")[:3]
idx = f"https://docs.gsplat.studio/whl/pt{tv[:4]}cu{cu}"
print(f"env: torch {tv} | cuda {torch.version.cuda}")
print(f"trying prebuilt wheel index: {idx}")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "gsplat", "--index-url", idx])

# validate the wheel actually loaded its CUDA extension; else fall back
fast = False
try:
    import gsplat.cuda._wrapper as _w
    fast = getattr(_w, "_C", None) is not None
except Exception:
    fast = False

print("prebuilt gsplat OK - skipping the 10-15 min compile!" if fast
      else "no prebuilt wheel matched - building gsplat from source (10-15 min, silent = normal)")

# install everything else (torch is preinstalled on Kaggle; this adds viser/tyro/lpips/etc.
# and, if the fast path failed, builds gsplat from source exactly as before)
!pip install -q -r requirements.txt

In [ ]:
# 2. Download + format the D-NeRF 'mutant' dataset (fully automatic)
!PYTHONPATH=/kaggle/working/SparseChron python scripts/prepare_dnerf.py --scene mutant --out /kaggle/working/data


In [ ]:
# 3. Download the trained checkpoint from the GitHub release (NO TRAINING, NO dataset)
#    Two models live at github.com/rajhodedara/SparseChron/releases/tag/trained-checkpoints:
#      - retrained (default): compact model, official metrics PSNR 29.17
#      - august: 347k Gaussians, dramatic motion (switch via cell 3b)
import os, urllib.request, shutil

os.makedirs("/kaggle/working/outputs_mutant", exist_ok=True)
BASE = "https://github.com/rajhodedara/SparseChron/releases/download/trained-checkpoints"

for fname in ["checkpoint_retrained_30000.ckpt", "scene_transform.json"]:
    dst = "/kaggle/working/outputs_mutant/" + ("checkpoint_30000.ckpt" if fname.startswith("checkpoint_") else fname)
    print(f"downloading {fname} ...")
    urllib.request.urlretrieve(f"{BASE}/{fname}", dst)
    print(f"  -> {dst} ({os.path.getsize(dst)/1e6:.1f} MB)")

import torch
state = torch.load("/kaggle/working/outputs_mutant/checkpoint_30000.ckpt", map_location="cpu", weights_only=False)
print(f"\nIteration: {state.get('iteration', 'n/a')} | Gaussians: {state['model_state_dict']['_positions'].shape[0]:,}")

print("scene_transform.json in place - eval aligned. Ready!")

In [ ]:
# 3b. (OPTIONAL) Switch to the AUGUST model - the dramatic-motion one
#     (347k Gaussians, hunched -> standing -> leaping). Downloads from the same
#     GitHub release; checks out the August-era code so colors render correctly.
#     After this cell, re-run the LAST cell (viewer) - it auto-picks
#     checkpoint_90000.ckpt as the newest.
import os, urllib.request

dst = "/kaggle/working/outputs_mutant/checkpoint_90000.ckpt"
if not os.path.exists(dst):
    print("downloading august model (237 MB)...")
    urllib.request.urlretrieve(
        "https://github.com/rajhodedara/SparseChron/releases/download/trained-checkpoints/checkpoint_august_30000.ckpt",
        dst)
print(f"august model in place ({os.path.getsize(dst)/1e6:.0f} MB)")

!cd /kaggle/working/SparseChron && git checkout ce7fb83
!pkill -f interactive_viewer
!pkill -f localtunnel
print("\nAugust model + August code staged. Re-run the LAST cell (viewer)!")

In [ ]:
# 4. Evaluate the NEWEST checkpoint (never a hardcoded filename)
import glob, json

ckpt = sorted(glob.glob("/kaggle/working/outputs_mutant/checkpoint_*.ckpt"))[-1]
print(f"Evaluating: {ckpt}")

!PYTHONPATH=/kaggle/working/SparseChron python scripts/evaluate.py --checkpoint-path {ckpt} --dataset-path /kaggle/working/data/dnerf_mutant --output-dir /kaggle/working/eval_mutant --is-4d

print("\n--- metrics.json ---")
with open("/kaggle/working/eval_mutant/metrics.json") as f:
    print(json.dumps(json.load(f), indent=2))


In [ ]:
# 5. Zip renders + metrics for download (Kaggle Output tab)
!zip -q -r /kaggle/working/mutant_renders.zip /kaggle/working/eval_mutant
print("Ready: /kaggle/working/mutant_renders.zip")


In [ ]:
# 6. Turn the rendered novel views into an MP4
!ffmpeg -y -framerate 30 -i /kaggle/working/eval_mutant/renders/render_%05d.png -c:v libx264 -pix_fmt yuv420p /kaggle/working/mutant_video.mp4


## 7. Interactive 4D Viewer (Viser)

Run the next cell, then:

1. Wait ~20 s while the model loads.
2. Click the `https://xxxx.loca.lt` URL printed at the bottom of the cell output.
3. localtunnel asks for a **password** — that is this machine's public IP, printed at the top of the cell output.
4. Scrub the **time slider** (or press play) to watch the 4D deformation animate.


In [ ]:
# 7. Launch the 4D Viser viewer + public tunnel
#    SKIPS ITSELF during Save & Run All (batch) - localtunnel never exits, and a
#    commit only saves its output when ALL cells finish. Run this cell manually
#    in an INTERACTIVE session to get the viewer + loca.lt link.
import os

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    print("[Save & Run All] Skipping the live viewer (it would block the commit forever).")
    print("To use the viewer: open an interactive session, run cells up to here, then this cell.")
else:
    !curl -s ifconfig.me; echo ""

    import glob, subprocess, time

    ckpt = sorted(glob.glob("/kaggle/working/outputs_mutant/checkpoint_*.ckpt"))[-1]

    subprocess.Popen([
        "python",
        "/kaggle/working/SparseChron/scripts/interactive_viewer.py",
        "--checkpoint", ckpt,
        "--port", "8080",
    ])
    time.sleep(15)  # give the viewer time to load the model

    print("Opening public tunnel...")
    !npx -y localtunnel --port 8080
